## PCS956 time series companion C: anomaly detection, concept drift, multivariate dependence, ML with context, and explainability

This companion notebook provides **code templates** for topics
discussed in lecture TS3:

- anomaly detection based on residuals and thresholds;
- concept drift monitoring and simple retraining strategies;
- multivariate dependence and cross-correlation;
- supervised learning on lagged plus exogenous/context features;
- simple feature-importance based explainability for tree-based models;
- scaffolding for simulation-based mini-projects, including synthetic data.

Each **code cell is self-contained**: it includes its own imports and helper functions,
so you can copy it into a fresh notebook without needing other cells. Illustrative
examples and extended discussion live in the TS3 lecture notebook.

You are expected to:

- adapt these templates to your own dataset or simulations;
- define clear train/validation/test splits, with gaps where appropriate;
- compare anomaly and drift methods against simple baselines (from Companions A and B);
- be explicit about the limitations of each method and the data.

The examples in this companion use simulated data. If you want to use
one of the templates for your mini-project or other applied work, you
must replace the simulation step with code that reads your time series
(see Companion A).


### 1. Residual-based anomaly flags for a univariate series

In many applications, anomalies are observations that are **implausible under some
notion of 'normal behaviour'**. There are several ways to formalise this:

- **Model-free rules**, for example:
  - values outside a physically meaningful range (for example negative counts, or
    heart rates above a plausible maximum),
  - jumps or local changes that are too fast to be realistic for the system
    (for example GPS altitude changes corresponding to jumping off a cliff when
    no such event is possible),
  - unusually large first differences relative to typical variability.

- **Model-based residuals**, where we:
  - fit a simple model to 'normal' data (for example a baseline forecasting model),
  - define anomalies as points where the residuals (observed minus predicted) are
    unusually large, given the estimated noise level.

The template in this section focuses on the **model-based residual** view, because it
fits naturally with the forecasting workflow in Companions B and C. It should be
seen as one practical approach among many, not as a universal definition of anomalies.

You can combine this with simpler checks on levels and differences: for example, you
might mark any observation as anomalous if it has both a large residual *and* a
physically implausible jump compared to the previous few points.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA

plt.style.use('seaborn-v0_8')


def fit_simple_arima(train, order=(1, 0, 0)):
    """
    Fit a simple ARIMA(p,d,q) model to 'train' for use as a baseline
    in anomaly detection.

    Parameters
    ----------
    train : pd.Series
        Univariate time series.
    order : tuple
        (p, d, q) order for ARIMA.

    Returns
    -------
    model, results : statsmodels ARIMA and fitted results
    """
    model = ARIMA(train, order=order)
    results = model.fit()
    return model, results


def multi_step_residuals(model_results, series):
    """
    Compute multi-step-ahead forecast residuals on a given series using a
    fitted ARIMA or AR model, forecasting from the end of the training sample.

    Parameters
    ----------
    model_results : statsmodels results object
        Fitted ARIMA or AR model, trained on an earlier window.
    series : pd.Series
        Univariate time series on which residuals should be computed.
        Typically a later evaluation window.

    Returns
    -------
    residuals : pd.Series
        Forecast residuals aligned with series.index.
    """
    forecast = model_results.forecast(steps=len(series))
    forecast = pd.Series(forecast, index=series.index, name='forecast')

    residuals = series - forecast
    residuals.name = 'residual'
    return residuals


def flag_anomalies(residuals, k=3.0):
    """
    Flag anomalies where |residual| exceeds k times the residual standard deviation.

    Parameters
    ----------
    residuals : pd.Series
        Residual series from a period believed to represent normal behaviour.
    k : float
        Threshold multiplier for standard deviation (for example k=3.0).

    Returns
    -------
    flags : pd.Series of bool
        True where an anomaly is flagged, False otherwise.
    """
    sigma = residuals.std(ddof=1)
    if sigma == 0 or np.isnan(sigma):
        raise ValueError(
            'Residual standard deviation is zero or NaN; cannot form a scale-based threshold.'
        )
    flags = residuals.abs() > k * sigma
    flags.name = 'anomaly_flag'
    return flags


def plot_anomalies(series, residuals, flags, k=3.0):
    """
    Plot a series with anomaly flags based on residuals.

    Parameters
    ----------
    series : pd.Series
        Original series.
    residuals : pd.Series
        Residual series.
    flags : pd.Series of bool
        Anomaly flags.
    k : float
        Threshold multiplier used for residuals.
    """
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    # Top: series with anomaly markers
    axes[0].plot(series.index, series.values, color='tab:blue', label='Series')
    axes[0].scatter(
        series.index[flags],
        series.values[flags],
        color='tab:red',
        label='Anomalies',
        zorder=3,
    )
    axes[0].set_title('Series with residual-based anomaly flags')
    axes[0].set_ylabel('Value')
    axes[0].legend(fontsize=8)

    # Bottom: residuals with threshold lines
    sigma = residuals.std(ddof=1)
    thr = k * sigma
    axes[1].plot(residuals.index, residuals.values, color='tab:orange', label='Residuals')
    axes[1].axhline(thr, color='grey', linestyle='--', linewidth=0.8, label=f'+{k}σ')
    axes[1].axhline(-thr, color='grey', linestyle='--', linewidth=0.8, label=f'-{k}σ')
    axes[1].set_title('Residuals and threshold')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Residual')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

Usage pattern:

1. Choose a training window where behaviour is believed to be relatively normal.
2. Fit a simple ARIMA model:

   ```python
   model, results = fit_simple_arima(train_series, order=(1, 0, 0))
   ```

3. Compute residuals on a later window:

   ```python
   resid = multi_step_residuals(results, eval_series)
   ```

4. Flag anomalies:

   ```python
   flags = flag_anomalies(resid, k=3.0)
   ```

5. Visualise:

   ```python
   plot_anomalies(eval_series, resid, flags, k=3.0)
   ```

Threshold choices are subjective and should be documented and
justified in your mini-project. Remember that the residual standard
deviation is estimated from the residuals you pass in (for example
from a calibration window); if the residual scale changes over time
due to drift, fixed thresholds may become unreliable.


**Example: residual-based anomalies on a synthetic series with a spike**

The following self-contained example simulates an AR(1) series, injects a single large
spike as an anomaly, and applies the residual-based anomaly detector from a 'normal'
training window to a later evaluation window.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA

plt.style.use('seaborn-v0_8')


def fit_simple_arima(train, order=(1, 0, 0)):
    model = ARIMA(train, order=order)
    results = model.fit()
    return model, results


def multi_step_residuals(model_results, series):
    forecast = model_results.forecast(steps=len(series))
    forecast = pd.Series(forecast, index=series.index, name='forecast')
    residuals = series - forecast
    residuals.name = 'residual'
    return residuals


def flag_anomalies(residuals, k=3.0):
    sigma = residuals.std(ddof=1)
    if sigma == 0 or np.isnan(sigma):
        raise ValueError(
            'Residual standard deviation is zero or NaN; cannot form a scale-based threshold.'
        )
    flags = residuals.abs() > k * sigma
    flags.name = 'anomaly_flag'
    return flags


def plot_anomalies(series, residuals, flags, k=3.0):
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(series.index, series.values, color='tab:blue', label='Series')
    axes[0].scatter(
        series.index[flags],
        series.values[flags],
        color='tab:red',
        label='Anomalies',
        zorder=3,
    )
    axes[0].set_title('Series with residual-based anomaly flags')
    axes[0].set_ylabel('Value')
    axes[0].legend(fontsize=8)

    sigma = residuals.std(ddof=1)
    thr = k * sigma
    axes[1].plot(residuals.index, residuals.values, color='tab:orange', label='Residuals')
    axes[1].axhline(thr, color='grey', linestyle='--', linewidth=0.8, label=f'+{k}σ')
    axes[1].axhline(-thr, color='grey', linestyle='--', linewidth=0.8, label=f'-{k}σ')
    axes[1].set_title('Residuals and threshold')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Residual')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


# 1. Simulate a simple AR(1) series
rng = np.random.default_rng(0)
n_total = 300
phi = 0.7
noise = rng.normal(scale=1.0, size=n_total)
x = np.zeros(n_total)
for t in range(1, n_total):
    x[t] = phi * x[t - 1] + noise[t]

# 2. Inject a simple anomaly: add a large spike at one time point
x[220] += 8.0

dates = pd.date_range(start='2000-01-01', periods=n_total, freq='D')
series = pd.Series(x, index=dates, name='x')

# 3. Define training (= normal) and evaluation windows
train = series.iloc[:200]
eval_series = series.iloc[200:]

# 4. Fit model on normal period and compute residuals on evaluation period
_, results = fit_simple_arima(train, order=(1, 0, 0))
resid = multi_step_residuals(results, eval_series)

# 5. Flag anomalies and visualise
flags = flag_anomalies(resid, k=3.0)
plot_anomalies(eval_series, resid, flags, k=3.0)

### 2. Concept drift monitoring via block-wise performance

**Concept drift** refers to changes in the
underlying behaviour of the process we are modelling. This can
include:

- shifts in the distribution of the target (for example changes in mean, variance,
  or seasonality);
- changes in how predictors relate to the target (for example a feature that used
  to be informative stops being useful);
- broader changes in the data-generating mechanism (for example new policies,
  technologies, or user behaviours).

When such changes occur, a model trained on past data can become
misaligned with current behaviour, and its predictive performance may
deteriorate. Monitoring model performance over time is therefore a
practical way to track drift: persistent drops or trends in error
metrics can be treated as signals that the underlying behaviour has
changed enough to warrant re-inspection or retraining.

The template below implements a simple block-wise performance monitor
using an expanding training window and fixed-size evaluation blocks.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA

plt.style.use('seaborn-v0_8')


def arima_model_fn(train, order=(1, 0, 0)):
    """
    Small wrapper to fit ARIMA for use with block_forecast_evaluation.
    """
    model = ARIMA(train, order=order)
    results = model.fit()
    return results


def block_forecast_evaluation(series, block_size, model_fn, model_kwargs=None):
    """
    Evaluate a forecasting model on successive non-overlapping blocks of a series.

    Parameters
    ----------
    series : pd.Series
        Univariate time series.
    block_size : int
        Number of observations per evaluation block.
    model_fn : callable
        Function that takes a training series and returns a fitted model with a
        .forecast(steps=...) method.
    model_kwargs : dict or None
        Optional keyword arguments for model_fn.

    Returns
    -------
    results_df : pd.DataFrame
        DataFrame with columns: 'block_start', 'block_end', 'MAE', 'RMSE', 'R2'
        (with 'R2' set to NaN when the block has zero variance).
    """
    if model_kwargs is None:
        model_kwargs = {}

    values = []
    n = len(series)

    # Expanding training window
    for start in range(block_size, n - block_size + 1, block_size):
        train = series.iloc[:start]
        test = series.iloc[start:start + block_size]

        model = model_fn(train, **model_kwargs)
        forecast = model.forecast(steps=len(test))

        y_true = test.values
        y_pred = np.asarray(forecast)

        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        if np.var(y_true) > 0:
            r2 = r2_score(y_true, y_pred)
        else:
            r2 = np.nan

        values.append({
            'block_start': test.index[0],
            'block_end': test.index[-1],
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
        })

    results_df = pd.DataFrame(values)
    return results_df


def plot_block_performance(results_df, metric='RMSE'):
    """
    Plot performance metric over successive blocks.

    Parameters
    ----------
    results_df : pd.DataFrame
        DataFrame with columns: 'block_start', 'block_end', 'MAE', 'RMSE', 'R2'
        (with 'R2' set to NaN when the block has zero variance).
    metric : str
        Column name in results_df to plot (for example 'RMSE', 'MAE', 'R2').
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(results_df['block_end'], results_df[metric], marker='o', color='tab:blue')
    ax.set_title(f'Performance over time ({metric})')
    ax.set_xlabel('Block end time')
    ax.set_ylabel(metric)
    plt.tight_layout()
    plt.show()

Usage pattern:

- Decide on a block size (for example 30 days, 12 months).
- Evaluate performance over time:

  ```python
  results_df = block_forecast_evaluation(
      series,
      block_size=30,
      model_fn=arima_model_fn,
      model_kwargs={'order': (1, 0, 0)},
  )
  ```

- Plot performance:

  ```python
  plot_block_performance(results_df, metric='RMSE')
  ```

Drops or trends in performance can be treated as potential drift
signals. In simulation-based mini-projects, you can generate series
with known drift and check that performance curves respond as
expected.


**Example: concept drift in an AR(1) series with a level shift**

This example simulates an AR(1) series with a level shift halfway through, then uses
block-wise expanding-window evaluation to show how RMSE changes over time. We first
plot the series itself (with the change-point marked), then the RMSE by block.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8')


def simulate_ar1_with_shift(
    n,
    phi=0.7,
    sigma1=1.0,
    sigma2=1.0,
    level1=0.0,
    level2=2.0,
    change_point=None,
    seed=123,
):
    rng = np.random.default_rng(seed)
    if change_point is None:
        change_point = n
    change_point = max(1, min(change_point, n))

    noise1 = rng.normal(loc=0.0, scale=sigma1, size=change_point)
    noise2 = rng.normal(loc=0.0, scale=sigma2, size=n - change_point)

    x = np.zeros(n)
    for t in range(1, change_point):
        x[t] = level1 + phi * (x[t - 1] - level1) + noise1[t]
    for t in range(change_point, n):
        x[t] = level2 + phi * (x[t - 1] - level2) + noise2[t - change_point]

    dates = pd.date_range(start='2000-01-01', periods=n, freq='MS')
    return pd.Series(x, index=dates, name='sim_ar1_shift')


def arima_model_fn(train, order=(1, 0, 0)):
    model = ARIMA(train, order=order)
    results = model.fit()
    return results


def block_forecast_evaluation(series, block_size, model_fn, model_kwargs=None):
    if model_kwargs is None:
        model_kwargs = {}

    values = []
    n = len(series)

    for start in range(block_size, n - block_size + 1, block_size):
        train = series.iloc[:start]
        test = series.iloc[start:start + block_size]

        model = model_fn(train, **model_kwargs)
        forecast = model.forecast(steps=len(test))

        y_true = test.values
        y_pred = np.asarray(forecast)

        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        if np.var(y_true) > 0:
            r2 = r2_score(y_true, y_pred)
        else:
            r2 = np.nan

        values.append({
            'block_start': test.index[0],
            'block_end': test.index[-1],
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
        })

    return pd.DataFrame(values)


def plot_block_performance(results_df, metric='RMSE'):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(results_df['block_end'], results_df[metric], marker='o', color='tab:blue')
    ax.set_title(f'Performance over time ({metric})')
    ax.set_xlabel('Block end time')
    ax.set_ylabel(metric)
    plt.tight_layout()
    plt.show()


# Simulate series with a level shift
n = 300
change_point = 150
series_sim = simulate_ar1_with_shift(
    n=n,
    phi=0.7,
    sigma1=1.0,
    sigma2=1.0,
    level1=0.0,
    level2=3.0,
    change_point=change_point,
    seed=42,
)

# Plot the simulated series with the change-point marked
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(series_sim.index, series_sim.values, color='tab:blue', label='Simulated AR(1) with shift')
ax.axvline(series_sim.index[change_point], color='red', linestyle='--', label='Change-point')
ax.set_title('Simulated AR(1) series with level shift')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Block-wise evaluation with an ARIMA(1,0,0) model
results_df = block_forecast_evaluation(
    series_sim,
    block_size=30,
    model_fn=arima_model_fn,
    model_kwargs={'order': (1, 0, 0)},
)

plot_block_performance(results_df, metric='RMSE')

### 2.1 Optional: using `auto_arima` to select ARIMA order

In the examples above we used an ARIMA(1, 0, 0) model. This was a deliberate simplification:
it keeps the focus on residuals, drift, and evaluation, without adding the extra complexity
of model selection.

In realistic applications we do not know the true ARIMA order. A common strategy is to let a
library search over a small set of candidate orders and choose the one that optimises an
information criterion such as AIC or BIC. This is still not automatic truth, but it provides a
transparent starting point.

The `pmdarima` package provides an `auto_arima` function that:

- tests differencing orders $d$ (and seasonal differencing if requested);
- explores a grid of $(p, d, q)$ (and optionally seasonal $(P, D, Q, m)$);
- selects a model using an information criterion (AIC by default).

The template below shows a minimal usage pattern on a synthetic AR(2) series. You can adapt
the ranges and options for your own data. This is intended as a convenience tool, not a
replacement for domain knowledge or careful diagnostics.

> Note: you will need to install `pmdarima` (for example with `pip install pmdarima`) to run
> this section.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pmdarima as pm

plt.style.use('seaborn-v0_8')


def fit_auto_arima(series):
    """
    Example: use pmdarima's auto_arima to select an ARIMA order.

    This is a convenience wrapper with modest bounds. Adjust the ranges and options
    for your own applications (for example enable seasonality where appropriate).
    """
    model = pm.auto_arima(
        series,
        start_p=0,
        start_q=0,
        max_p=3,
        max_q=3,
        d=None,                 # let auto_arima test d via unit root tests
        seasonal=False,         # set to True and specify m for seasonal data
        information_criterion='aic',
        stepwise=True,
        suppress_warnings=True,
        error_action='ignore',
        random_state=0,
    )
    return model


# Example: simulate an AR(2) series and let auto_arima choose (p, d, q)

rng = np.random.default_rng(0)
n = 300
eps = rng.normal(scale=1.0, size=n)
x = np.zeros(n)
for t in range(2, n):
    # AR(2) with coefficients 0.6 and -0.2
    x[t] = 0.6 * x[t - 1] - 0.2 * x[t - 2] + eps[t]

idx = pd.date_range(start='2000-01-01', periods=n, freq='D')
series_ar2 = pd.Series(x, index=idx, name='sim_ar2')

# Fit auto_arima
auto_model = fit_auto_arima(series_ar2)

print('Selected order (p, d, q):', auto_model.order)
print('AIC of selected model:', auto_model.aic())

# Optional: quick visual check of fit on the last 50 points
n_train = 250
train = series_ar2.iloc[:n_train]
test = series_ar2.iloc[n_train:]

auto_model_fit = auto_model.fit(train)
y_pred = auto_model_fit.predict(n_periods=len(test))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(series_ar2.index, series_ar2.values, label='Series', color='tab:blue')
ax.plot(test.index, y_pred, label='auto_arima forecast (test window)', color='tab:orange')
ax.axvline(test.index[0], color='grey', linestyle='--', linewidth=0.8, label='Train/test split')
ax.set_title('AR(2) example with auto_arima-selected ARIMA model')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3. Multivariate dependence: cross-correlation and Granger-style tests

For multivariate time series it is often useful to understand how
variables move together over time and whether one series helps to
predict another.

- **Cross-correlation functions (CCFs)** measure (linear) correlation between two
  series at different lags:
  - a positive lag (for example lag = +3) asks: how well does the past of series X
    line up with the present of series Y (X leading Y)?
  - a negative lag asks the opposite (Y leading X).
  - this is symmetric and descriptive: it tells you about linear dependence patterns,
    not about causal direction.

- **Granger-style tests** ask a predictive question:
  - does including past values of series X improve the prediction of series Y,
    beyond using past values of Y alone?
  - if so, we say 'X Granger-causes Y' in a predictive sense.
  - this is a stronger statement than simple correlation, but still about prediction,
    not about true causal mechanisms in the physical or social sense.

The templates below provide:

- a helper to compute and plot cross-correlation between two series;
- a small wrapper to run Granger-style predictive tests using statsmodels.

They are intended as **exploratory tools**. Interpretation should be
cautious, and ideally supported by domain knowledge and additional
modelling.

Note that cross-correlations at large lags are based on fewer overlapping observations and can be
unstable. It is often sensible to restrict attention to moderate lags where a reasonable amount of
data is available.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import grangercausalitytests

plt.style.use('seaborn-v0_8')


def cross_correlation(series_x, series_y, max_lag=40):
    """
    Compute sample cross-correlation between series_x and lagged versions of series_y.

    Parameters
    ----------
    series_x, series_y : pd.Series
        Two time series aligned on the same index and observed at equal time steps.
    max_lag : int
        Maximum lag (in time steps) to consider.

    Returns
    -------
    lags : np.ndarray
        Array of lag values (positive and negative).
    ccf : np.ndarray
        Sample cross-correlation values for each lag.
    """
    if len(series_x) != len(series_y):
        raise ValueError('series_x and series_y must have the same length and aligned indices')

    n = len(series_x)
    if max_lag >= n:
        raise ValueError('max_lag must be smaller than the series length')

    x = series_x.values - series_x.values.mean()
    y = series_y.values - series_y.values.mean()

    lags = np.arange(-max_lag, max_lag + 1)
    ccf = np.zeros_like(lags, dtype=float)

    for i, lag in enumerate(lags):
        if lag < 0:
            # y leads x (negative lag)
            ccf[i] = np.corrcoef(x[-lag:], y[:n + lag])[0, 1]
        elif lag > 0:
            # x leads y (positive lag)
            ccf[i] = np.corrcoef(x[:n - lag], y[lag:])[0, 1]
        else:
            ccf[i] = np.corrcoef(x, y)[0, 1]

    return lags, ccf


def plot_cross_correlation(series_x, series_y, max_lag=40, label_x='X', label_y='Y'):
    """
    Plot cross-correlation between two series.

    Parameters
    ----------
    series_x, series_y : pd.Series
    max_lag : int
    label_x, label_y : str
        Labels for the series.
    """
    lags, ccf = cross_correlation(series_x, series_y, max_lag=max_lag)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.stem(lags, ccf)
    ax.set_title(f'Cross-correlation between {label_x} and {label_y}')
    ax.set_xlabel(f'Lag (positive: {label_x} leads {label_y})')
    ax.set_ylabel('Correlation')
    plt.tight_layout()
    plt.show()


def granger_test(df, cause_col, effect_col, maxlag=4):
    """
    Run Granger-style predictive causality tests using statsmodels.tsa.stattools.grangercausalitytests.

    The `grangercausalitytests` function expects a two-column array with the potential effect in the
    first column and the potential cause in the second, which is why `df[[effect_col, cause_col]]` is
    used inside `granger_test`.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the two series as columns.
    cause_col : str
        Column name of potential cause series.
    effect_col : str
        Column name of effect series.
    maxlag : int
        Maximum lag to test.

    Returns
    -------
    results : dict
        Output from grangercausalitytests.
    """
    data = df[[effect_col, cause_col]].dropna()
    results = grangercausalitytests(data, maxlag=maxlag, verbose=False)
    return results

Usage pattern:

- Align two series on a common index and consider appropriate transforms (for example
  differences or residuals):

  ```python
  plot_cross_correlation(series_x, series_y, max_lag=40, label_x='X', label_y='Y')
  ```

- For Granger-style tests:

  ```python
  df_xy = pd.DataFrame({'Y': series_y, 'X': series_x})
  results = granger_test(df_xy, cause_col='X', effect_col='Y', maxlag=4)
  ```

Granger-style predictive influence is about forecasting improvement, not definitive
causal effect. Use these tools as initial probes, not final answers.


**Example: cross-correlation for a synthetic lead–lag relationship**

The following example constructs two synthetic series where one is designed to respond
(with lag) to the other. We first plot the two series to see the lead–lag structure,
and then plot the cross-correlation function, which highlights the same pattern.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')


def cross_correlation(series_x, series_y, max_lag=40):
    if len(series_x) != len(series_y):
        raise ValueError('series_x and series_y must have the same length and aligned indices')

    n = len(series_x)
    if max_lag >= n:
        raise ValueError('max_lag must be smaller than the series length')

    x = series_x.values - series_x.values.mean()
    y = series_y.values - series_y.values.mean()

    lags = np.arange(-max_lag, max_lag + 1)
    ccf = np.zeros_like(lags, dtype=float)

    for i, lag in enumerate(lags):
        if lag < 0:
            # y leads x (negative lag)
            ccf[i] = np.corrcoef(x[-lag:], y[:n + lag])[0, 1]
        elif lag > 0:
            # x leads y (positive lag)
            ccf[i] = np.corrcoef(x[:n - lag], y[lag:])[0, 1]
        else:
            ccf[i] = np.corrcoef(x, y)[0, 1]

    return lags, ccf


def plot_cross_correlation(series_x, series_y, max_lag=40, label_x='X', label_y='Y'):
    lags, ccf = cross_correlation(series_x, series_y, max_lag=max_lag)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.stem(lags, ccf)
    ax.set_title(f'Cross-correlation between {label_x} and {label_y}')
    ax.set_xlabel(f'Lag (positive: {label_x} leads {label_y})')
    ax.set_ylabel('Correlation')
    plt.tight_layout()
    plt.show()


# Example: synthetic lead–lag relationship

rng = np.random.default_rng(0)
n = 300

# X_t is AR(1) noise
eps = rng.normal(scale=1.0, size=n)
x = np.zeros(n)
for t in range(1, n):
    x[t] = 0.8 * x[t - 1] + eps[t]

# Y_t responds to X_{t-2} plus its own noise
y = np.zeros(n)
for t in range(2, n):
    y[t] = 0.5 * x[t - 2] + rng.normal(scale=1.0)

idx = pd.date_range(start='2000-01-01', periods=n, freq='D')
series_x = pd.Series(x, index=idx, name='X')
series_y = pd.Series(y, index=idx, name='Y')

# Plot the two series
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(series_x.index, series_x.values, label='X (driver)', color='tab:blue')
ax.plot(series_y.index, series_y.values, label='Y (response)', color='tab:orange', alpha=0.8)
ax.set_title('Synthetic lead–lag system: X drives Y with a lag')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Plot the cross-correlation function
plot_cross_correlation(series_x, series_y, max_lag=10, label_x='X', label_y='Y')

**Example: Granger-style test on a synthetic bivariate system**

We reuse the synthetic series X and Y from the previous example, where
X was constructed to drive Y with a lag. We now apply a simple
Granger-style predictive test to see whether including past X improves
prediction of Y, beyond using past Y alone.

For each lag $l = 1, 2, \dots, \text{maxlag}$, the `grangercausalitytests` function runs an
F-test for the null hypothesis:

- $H_0$: past values of X up to lag $l$ do not help to predict Y, beyond using past Y alone;
- $H_1$: past values of X up to lag $l$ do help to predict Y.

We typically fix a significance level, for example $\alpha = 0.05$:

- if $p\text{-value} < \alpha$ we reject $H_0$ at lag $l$ and conclude that X Granger-causes Y
  (in a predictive sense) at that lag;
- if $p\text{-value} \ge \alpha$ we do not reject $H_0$ at lag $l$.

This is about short-term predictive value, not about true causal
mechanisms. In this synthetic setting, X is designed to influence Y,
so we expect to see evidence against $H_0$ at some lags, but this
should be interpreted as a predictive relationship, not as a full
causal story about a real system.


In [ ]:
import numpy as np
import pandas as pd
import warnings

from statsmodels.tsa.stattools import grangercausalitytests

# Suppress the FutureWarning about the 'verbose' argument in grangercausalitytests
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="statsmodels.tsa.stattools",
)


def granger_test(df, cause_col, effect_col, maxlag=4):
    """
    Run Granger-style tests of whether 'cause_col' helps to predict 'effect_col'.

    Returns the full statsmodels results dict.
    """
    data = df[[effect_col, cause_col]].dropna()
    results = grangercausalitytests(data, maxlag=maxlag, verbose=False)
    return results


# Simulate a simple bivariate system: X_t influences Y_t with a lag
rng = np.random.default_rng(1)
n = 300

# X_t is AR(1) noise
eps_x = rng.normal(scale=1.0, size=n)
X = np.zeros(n)
for t in range(1, n):
    X[t] = 0.7 * X[t - 1] + eps_x[t]

# Y_t depends on its own lag and past X
eps_y = rng.normal(scale=1.0, size=n)
Y = np.zeros(n)
for t in range(1, n):
    Y[t] = 0.5 * Y[t - 1] + 0.4 * X[t - 1] + eps_y[t]

idx = pd.date_range(start='2000-01-01', periods=n, freq='D')
df_xy = pd.DataFrame({'Y': Y, 'X': X}, index=idx)

# Run Granger-style tests: does X help to predict Y?
maxlag = 4
results_xy = granger_test(df_xy, cause_col='X', effect_col='Y', maxlag=maxlag)

# Extract p-values for the F-test at each lag, as plain Python types
pvals = {int(lag): float(res[0]['ssr_ftest'][1]) for lag, res in results_xy.items()}

alpha = 0.05
print('P-values for H0: "X does NOT Granger-cause Y" (F-test):')
for lag in range(1, maxlag + 1):
    p = pvals[lag]
    decision = "REJECT H0 (evidence that X Granger-causes Y)" if p < alpha else "do NOT reject H0"
    print(f"  lag {lag}: p = {p:.3e}  ->  {decision}")

### 4. Supervised learning with lagged plus exogenous features

Many forecasting and anomaly-detection problems use both lagged values and exogenous or
context variables (for example calendar effects, weather, or regime indicators).

This template extends the lagged-feature construction from Companion B to include exogenous
inputs.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor


def series_and_exogenous_to_supervised(target, exogenous_df, n_lags=10, horizon=1):
    """
    Build supervised learning data from a target series and exogenous features.

    For each time t where lagged target values and exogenous features are available,
    the input vector X_t contains:
    - previous n_lags target values (times t-1, t-2, ..., t-n_lags),
    - current exogenous features at time t,
    and the target y_t is the target at time t + horizon - 1.

    For example, horizon=1 gives one-step-ahead forecasting; horizon=3 uses exogenous features
    at time t and predicts target at time t+2.

    Parameters
    ----------
    target : pd.Series
        Target time series.
    exogenous_df : pd.DataFrame
        Exogenous features aligned with target.index.
    n_lags : int
    horizon : int

    Returns
    -------
    X, y, feature_names : np.ndarray, np.ndarray, list of str
    """
    if not target.index.equals(exogenous_df.index):
        raise ValueError('target and exogenous_df must have the same index (aligned in time)')

    y_vals = target.values
    exog_vals = exogenous_df.values
    n = len(target)
    X_list, y_list = [], []

    feature_names = [f'lag_{i}' for i in range(1, n_lags + 1)] + list(exogenous_df.columns)

    for t in range(n_lags, n - horizon + 1):
        # Input at time t uses target lags [t-1, ..., t-n_lags] and exogenous at t
        lag_block = y_vals[t - n_lags:t]
        exog_block = exog_vals[t]
        X_list.append(np.concatenate([lag_block, exog_block]))
        # Target is at time t + horizon - 1
        y_list.append(y_vals[t + horizon - 1])

    return np.array(X_list), np.array(y_list), feature_names


def rf_model_exog_fn(X_train, y_train):
    """
    Fit a RandomForestRegressor for lagged+exogenous features.
    """
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return model

Usage pattern:

- Construct a target series and an exogenous DataFrame with aligned indices:

  ```python
  target = my_df['y']
  exog = my_df[['dow', 'holiday', 'temp']]
  ```

- Build supervised data:

  ```python
  X, y, feature_names = series_and_exogenous_to_supervised(target, exog, n_lags=14, horizon=1)
  ```

- Split `(X, y)` into train/validation/test according to time (for example using the same
  positional split pattern as in Companion B), then fit and evaluate:

  ```python
  model = rf_model_exog_fn(X_train, y_train)
  y_pred = model.predict(X_test)
  ```

This template is intended for forecasting tasks with exogenous inputs; similar patterns
can be adapted for anomaly detection and drift monitoring.


**Example: daily demand forecasting with a weekend effect (RandomForest and Gradient Boosting)**

This example simulates a daily demand series with an AR(1) structure and a weekend uplift.
We fit two modern tree-ensemble models on lagged demand values plus a simple binary
'is_weekend' context feature:

- RandomForestRegressor (bagging-style trees),
- HistGradientBoostingRegressor (gradient boosting on histograms).

We first plot the simulated series, then compare predictions on the test period.

Depending on the random seed and hyperparameters, either the Random
Forest or the gradient boosting model may perform slightly better on
this toy dataset. Here the Random Forest has a lower MAE, but the main
point is that both models fit naturally into the same
lag-plus-exogenous pipeline.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

plt.style.use('seaborn-v0_8')


def series_and_exogenous_to_supervised(target, exogenous_df, n_lags=10, horizon=1):
    if not target.index.equals(exogenous_df.index):
        raise ValueError('target and exogenous_df must have the same index (aligned in time)')

    y_vals = target.values
    exog_vals = exogenous_df.values
    n = len(target)
    X_list, y_list = [], []

    feature_names = [f'lag_{i}' for i in range(1, n_lags + 1)] + list(exogenous_df.columns)

    for t in range(n_lags, n - horizon + 1):
        lag_block = y_vals[t - n_lags:t]
        exog_block = exog_vals[t]
        X_list.append(np.concatenate([lag_block, exog_block]))
        y_list.append(y_vals[t + horizon - 1])

    return np.array(X_list), np.array(y_list), feature_names


def rf_model_exog_fn(X_train, y_train):
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return model


def gbm_model_exog_fn(X_train, y_train):
    """
    Fit a histogram-based gradient boosting model on lagged+exogenous features.
    This is a modern tree-ensemble method available in scikit-learn.
    """
    model = HistGradientBoostingRegressor(
        learning_rate=0.1,
        max_iter=300,
        max_depth=None,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return model


# Example: daily demand with a weekend effect

rng = np.random.default_rng(0)
n_days = 365
dates = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
dow = dates.dayofweek  # 0=Monday, ..., 6=Sunday

# Simulate base demand with AR(1) structure
noise = rng.normal(scale=2.0, size=n_days)
demand = np.zeros(n_days)
for t in range(1, n_days):
    demand[t] = 0.7 * demand[t - 1] + noise[t]

# Add a weekend uplift
weekend_mask = (dow >= 5).astype(float)
demand = demand + 10.0 * weekend_mask

target = pd.Series(demand, index=dates, name='demand')
exog = pd.DataFrame({'is_weekend': weekend_mask}, index=dates)

# Plot the simulated series with weekends indicated
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(target.index, target.values, label='Demand', color='tab:blue')
ax.fill_between(
    target.index,
    target.values.min() - 1,
    target.values.max() + 1,
    where=weekend_mask.astype(bool),
    color='tab:orange',
    alpha=0.15,
    label='Weekend',
)
ax.set_title('Simulated daily demand with weekend uplift')
ax.set_xlabel('Date')
ax.set_ylabel('Demand')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Build supervised data
X, y, feature_names = series_and_exogenous_to_supervised(target, exog, n_lags=7, horizon=1)

# Simple time-based split: first 250 for training, rest for test
split = 250
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
test_index = target.index[7 + split:7 + len(y)]  # align with y_test

rf = rf_model_exog_fn(X_train, y_train)
gbm = gbm_model_exog_fn(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_gbm = gbm.predict(X_test)

print('RandomForest  Test MAE:', mean_absolute_error(y_test, y_pred_rf))
print('GradientBoost Test MAE:', mean_absolute_error(y_test, y_pred_gbm))

# Plot predictions vs actuals on the test period
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_index, y_test, label='Actual demand', color='black')
ax.plot(test_index, y_pred_rf, label='RF prediction', color='tab:blue', alpha=0.8)
ax.plot(test_index, y_pred_gbm, label='GBM prediction', color='tab:orange', alpha=0.8)
ax.set_title('Predictions vs actuals on test period')
ax.set_xlabel('Date')
ax.set_ylabel('Demand')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 5. Feature importance for tree-based models on lagged and context features

Explainability for tree-based models can start from feature importance scores. This template
provides:

- a helper to compute feature importances from a fitted RandomForest;
- a plotting function to visualise importances;
- an optional aggregation of importances for lagged vs exogenous features.

Note on 'impurity-based' importance

For tree-based models (for example RandomForest and gradient
boosting), each split in a tree is chosen to make the target values in
the child nodes more homogeneous than in the parent node. A measure of
how mixed or heterogeneous a node is is called an impurity measure
(for regression, this is often the variance of the target in that
node).

Impurity-based feature importance works by:

- looking at how much each split reduces impurity (for example how much the variance of the
  target decreases after the split), and
- attributing that reduction to the feature used in the split,
- then summing these reductions over all splits and all trees for each feature.

The built-in `feature_importances_` attribute in scikit-learn reports
these aggregated impurity reductions, normalised to sum to 1 across
features.

This is convenient and fast, but it has limitations. In particular, it
can over-favour features that have many distinct values (many possible
split points) compared to binary or low-cardinality features. For more
robust interpretability it is often useful to complement these scores
with permutation importance, which measures how much the model's error
increases when a feature is randomly shuffled in the test data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')


def feature_importance_df(model, feature_names):
    """
    Build a DataFrame of impurity-based feature importances for a fitted tree-based model.

    Parameters
    ----------
    model : fitted tree-based model with .feature_importances_
        (for example RandomForestRegressor or GradientBoostingRegressor).
    feature_names : list of str
        Names of the input features corresponding to the model inputs.

    Returns
    -------
    df : pd.DataFrame
        Columns: 'feature', 'importance'.
    """
    if not hasattr(model, 'feature_importances_'):
        raise AttributeError(
            'Model does not have attribute feature_importances_. '
            'Ensure you use a tree-based model such as RandomForestRegressor.'
        )
    importances = model.feature_importances_
    if len(importances) != len(feature_names):
        raise ValueError('Length of feature_names does not match model.feature_importances_')

    df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances,
    })
    df = df.sort_values('importance', ascending=False)
    return df


def plot_feature_importance(df, top_k=20):
    """
    Plot top-k feature importances from a DataFrame.

    Parameters
    ----------
    df : pd.DataFrame with columns 'feature' and 'importance'
    top_k : int
        Number of top features to show.
    """
    df_top = df.head(top_k)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(df_top['feature'][::-1], df_top['importance'][::-1], color='tab:blue')
    ax.set_title('Feature importances (top-k)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()


def aggregate_importance(df, lag_prefix='lag_'):
    """
    Aggregate feature importances into lagged vs exogenous groups.

    Parameters
    ----------
    df : pd.DataFrame
        Feature importance table.
    lag_prefix : str
        Prefix used for lag features.

    Returns
    -------
    agg : pd.DataFrame
        Aggregated importance by group.
    """
    total = df['importance'].sum()
    lag_mask = df['feature'].str.startswith(lag_prefix)
    lag_importance = df.loc[lag_mask, 'importance'].sum()
    exog_importance = total - lag_importance

    agg = pd.DataFrame({
        'group': ['lagged', 'exogenous'],
        'importance': [lag_importance, exog_importance],
    })
    agg['relative_importance'] = agg['importance'] / total
    return agg

Usage pattern:

- After fitting a RandomForest model on lagged+exogenous features, build a feature importance
  table:

  ```python
  df_imp = feature_importance_df(model, feature_names)
  plot_feature_importance(df_imp, top_k=20)
  ```

- Optionally aggregate lagged vs exogenous importance:

  ```python
  agg_imp = aggregate_importance(df_imp, lag_prefix='lag_')
  display(agg_imp)
  ```

This helps to explain which past values and context variables drive forecasts or anomaly
flags, and to connect ML behaviour to domain knowledge.


**Example: feature importance on lagged demand and weekend indicator**

We reuse the synthetic daily demand series with a weekend uplift, fit a RandomForest on
lagged demand plus an `is_weekend` feature, and inspect feature importances.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

plt.style.use('seaborn-v0_8')


def series_and_exogenous_to_supervised(target, exogenous_df, n_lags=7, horizon=1):
    if not target.index.equals(exogenous_df.index):
        raise ValueError('target and exogenous_df must have the same index (aligned in time)')

    y_vals = target.values
    exog_vals = exogenous_df.values
    n = len(target)
    X_list, y_list = [], []

    feature_names = [f'lag_{i}' for i in range(1, n_lags + 1)] + list(exogenous_df.columns)

    for t in range(n_lags, n - horizon + 1):
        lag_block = y_vals[t - n_lags:t]
        exog_block = exog_vals[t]
        X_list.append(np.concatenate([lag_block, exog_block]))
        y_list.append(y_vals[t + horizon - 1])

    return np.array(X_list), np.array(y_list), feature_names


def feature_importance_df(model, feature_names):
    if not hasattr(model, 'feature_importances_'):
        raise AttributeError(
            'Model does not have attribute feature_importances_. '
            'Ensure you use a tree-based model such as RandomForestRegressor.'
        )
    importances = model.feature_importances_
    if len(importances) != len(feature_names):
        raise ValueError('Length of feature_names does not match model.feature_importances_')

    df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    df = df.sort_values('importance', ascending=False)
    return df


def plot_feature_importance(df, top_k=20):
    df_top = df.head(top_k)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(df_top['feature'][::-1], df_top['importance'][::-1], color='tab:blue')
    ax.set_title('Feature importances (top-k)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()


# Simulate daily demand with weekend effect
rng = np.random.default_rng(0)
n_days = 365
dates = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
dow = dates.dayofweek

noise = rng.normal(scale=2.0, size=n_days)
demand = np.zeros(n_days)
for t in range(1, n_days):
    demand[t] = 0.7 * demand[t - 1] + noise[t]

weekend_mask = (dow >= 5).astype(float)
demand = demand + 10.0 * weekend_mask

target = pd.Series(demand, index=dates, name='demand')
exog = pd.DataFrame({'is_weekend': weekend_mask}, index=dates)

X, y, feature_names = series_and_exogenous_to_supervised(target, exog, n_lags=7, horizon=1)

split = 250
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

rf = RandomForestRegressor(n_estimators=300, random_state=0)
rf.fit(X_train, y_train)

df_imp = feature_importance_df(rf, feature_names)
plot_feature_importance(df_imp, top_k=10)

### 6. Simulation-based scaffolding for anomaly and drift studies

Simulation-based projects are explicitly acceptable for the time-series mini-project. This
section provides a self-contained pattern for:

- simulating an AR(1) series with a change in variance (noise level);
- evaluating an ARIMA model block-wise over time;
- plotting both the simulated series and the performance curve.

You can adapt this pattern to construct more complex scenarios (for example level shifts,
regime changes, or multivariate systems).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8')


def simulate_ar1_with_variance_change(
    n,
    phi=0.6,
    sigma1=1.0,
    sigma2=2.0,
    level=0.0,
    change_point=None,
    seed=123,
):
    """
    Simulate an AR(1) series with a change in noise variance at a given time.

    Parameters
    ----------
    n : int
        Total length of series.
    phi : float
        AR(1) coefficient.
    sigma1 : float
        Noise standard deviation before change_point.
    sigma2 : float
        Noise standard deviation after change_point.
    level : float
        Mean level throughout.
    change_point : int or None
        Index (0-based) at which the variance change occurs. If None, no change.
    seed : int
        Random seed.

    Returns
    -------
    series : pd.Series
        Simulated series with a DatetimeIndex (monthly) for convenience.
    """
    rng = np.random.default_rng(seed)
    if change_point is None:
        change_point = n
    change_point = max(1, min(change_point, n))

    noise1 = rng.normal(loc=0.0, scale=sigma1, size=change_point)
    noise2 = rng.normal(loc=0.0, scale=sigma2, size=n - change_point)

    x = np.zeros(n)
    x[0] = level
    for t in range(1, change_point):
        x[t] = level + phi * (x[t - 1] - level) + noise1[t]
    for t in range(change_point, n):
        x[t] = level + phi * (x[t - 1] - level) + noise2[t - change_point]

    dates = pd.date_range(start='2000-01-01', periods=n, freq='MS')
    series = pd.Series(x, index=dates, name='sim_ar1_var_change')
    return series


def block_forecast_evaluation_sim(series, block_size, order=(1, 0, 0)):
    """
    Evaluate an ARIMA model on successive non-overlapping blocks of a simulated series.

    Parameters
    ----------
    series : pd.Series
        Simulated series.
    block_size : int
        Number of observations per evaluation block.
    order : tuple
        ARIMA order.

    Returns
    -------
    results_df : pd.DataFrame
        Columns: 'block_start', 'block_end', 'MAE', 'RMSE', 'R2'.
    """
    values = []
    n = len(series)

    for start in range(block_size, n - block_size + 1, block_size):
        train = series.iloc[:start]
        test = series.iloc[start:start + block_size]

        model = ARIMA(train, order=order)
        results = model.fit()
        forecast = results.forecast(steps=len(test))

        y_true = test.values
        y_pred = np.asarray(forecast)

        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        if np.var(y_true) > 0:
            r2 = r2_score(y_true, y_pred)
        else:
            r2 = np.nan

        values.append({
            'block_start': test.index[0],
            'block_end': test.index[-1],
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
        })

    results_df = pd.DataFrame(values)
    return results_df


def plot_block_performance(results_df, metric='RMSE'):
    """
    Plot a performance metric over successive blocks.

    Parameters
    ----------
    results_df : pd.DataFrame
        DataFrame with columns: 'block_start', 'block_end', 'MAE', 'RMSE', 'R2'.
    metric : str
        Column name in results_df to plot (for example 'RMSE', 'MAE', 'R2').
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(results_df['block_end'], results_df[metric], marker='o', color='tab:blue')
    ax.set_title(f'Performance over time ({metric})')
    ax.set_xlabel('Block end time')
    ax.set_ylabel(metric)
    plt.tight_layout()
    plt.show()


# Example: AR(1) with a variance change and block-wise performance

n = 300
change_point = 150

series_sim = simulate_ar1_with_variance_change(
    n=n,
    phi=0.7,
    sigma1=1.0,
    sigma2=2.5,
    level=0.0,
    change_point=change_point,
    seed=42,
)

# Plot the simulated series with the variance-change point marked
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(series_sim.index, series_sim.values, color='tab:blue', label='Simulated AR(1)')
ax.axvline(series_sim.index[change_point], color='red', linestyle='--', label='Variance change')
ax.set_title('Simulated AR(1) series with variance change')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Block-wise evaluation with an ARIMA(1, 0, 0) model
results_sim = block_forecast_evaluation_sim(series_sim, block_size=30, order=(1, 0, 0))
plot_block_performance(results_sim, metric='RMSE')

Usage pattern:

- Simulate a drifting or variance-change series:

  ```python
  series_sim = simulate_ar1_with_variance_change(
      n=300,
      phi=0.7,
      sigma1=1.0,
      sigma2=2.5,
      level=0.0,
      change_point=150,
      seed=42,
  )
  ```

- Evaluate drift behaviour:

  ```python
  results_sim = block_forecast_evaluation_sim(series_sim, block_size=30, order=(1, 0, 0))
  ```

- Plot performance over time:

  ```python
  plot_block_performance(results_sim, metric='RMSE')
  ```


### 7. Walk-forward evaluation for ML models on lagged + exogenous features

For ML models on lagged+exogenous features (for example RandomForest or gradient boosting),
we can use a simple **walk-forward evaluation** scheme:

- start with an initial training window;
- repeatedly refit the model on all available past data;
- evaluate on the next block of observations;
- move the window forward and repeat.

This mirrors model maintenance under concept drift: as more data arrive, we update the model
and track performance over time. The function below is a generic walk-forward evaluator for
any regressor with a scikit-learn style `.fit` / `.predict` interface.


In [ ]:
import numpy as np
import pandas as pd


def walk_forward_evaluation(X, y, model_fn, initial_train, step, horizon):
    """
    Simple walk-forward evaluation for supervised models on time-ordered (X, y).

    Parameters
    ----------
    X : np.ndarray
        Feature matrix in temporal order.
    y : np.ndarray
        Target array in temporal order.
    model_fn : callable
        Function that takes (X_train, y_train) and returns a fitted model with .predict().
    initial_train : int
        Number of initial observations to train on.
    step : int
        Step size to advance the train/test boundary.
    horizon : int
        Number of observations in each test block.

    Returns
    -------
    results_df : pd.DataFrame with columns ['start', 'end', 'MAE', 'RMSE']
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    n = len(y)
    results = []

    start = initial_train
    while start + horizon <= n:
        X_train, y_train = X[:start], y[:start]
        X_test, y_test = X[start:start + horizon], y[start:start + horizon]

        model = model_fn(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        results.append({
            'start': start,
            'end': start + horizon - 1,
            'MAE': mae,
            'RMSE': rmse,
        })

        start += step

    return pd.DataFrame(results)

**Example: walk-forward evaluation of RandomForest on the weekend-demand data**

We recreate the lagged+exogenous feature matrix `(X, y)` from the weekend-demand example in
Section 4. We then:

- start with 200 days of training data,
- evaluate RandomForest on blocks of 30 days,
- move the boundary forward by 30 days each time,
- and plot how MAE and RMSE change over the walk-forward steps.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

plt.style.use('seaborn-v0_8')


def rf_model_exog_fn(X_train, y_train):
    """
    Fit a RandomForestRegressor on lagged+exogenous features.
    """
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return model


def series_and_exogenous_to_supervised(target, exogenous_df, n_lags=7, horizon=1):
    """
    Build supervised data (X, y) from a target series and exogenous features.
    """
    if not target.index.equals(exogenous_df.index):
        raise ValueError('target and exogenous_df must have the same index (aligned in time)')

    y_vals = target.values
    exog_vals = exogenous_df.values
    n = len(target)
    X_list, y_list = [], []

    feature_names = [f'lag_{i}' for i in range(1, n_lags + 1)] + list(exogenous_df.columns)

    for t in range(n_lags, n - horizon + 1):
        lag_block = y_vals[t - n_lags:t]
        exog_block = exog_vals[t]
        X_list.append(np.concatenate([lag_block, exog_block]))
        y_list.append(y_vals[t + horizon - 1])

    return np.array(X_list), np.array(y_list), feature_names


def walk_forward_evaluation(X, y, model_fn, initial_train, step, horizon):
    """
    Simple walk-forward evaluation for supervised models on time-ordered (X, y).
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    n = len(y)
    results = []

    start = initial_train
    while start + horizon <= n:
        X_train, y_train = X[:start], y[:start]
        X_test, y_test = X[start:start + horizon], y[start:start + horizon]

        model = model_fn(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        results.append({
            'start': start,
            'end': start + horizon - 1,
            'MAE': mae,
            'RMSE': rmse,
        })

        start += step

    return pd.DataFrame(results)


def plot_walk_forward_metrics(results_df):
    """
    Plot MAE and RMSE across walk-forward steps.
    """
    steps = np.arange(len(results_df))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, results_df['MAE'], marker='o', label='MAE', color='tab:blue')
    ax.plot(steps, results_df['RMSE'], marker='s', label='RMSE', color='tab:orange')
    ax.set_title('Walk-forward performance over blocks')
    ax.set_xlabel('Walk-forward step')
    ax.set_ylabel('Error')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


# Recreate the weekend-demand dataset and supervised matrix
rng = np.random.default_rng(0)
n_days = 365
dates = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
dow = dates.dayofweek

noise = rng.normal(scale=2.0, size=n_days)
demand = np.zeros(n_days)
for t in range(1, n_days):
    demand[t] = 0.7 * demand[t - 1] + noise[t]

weekend_mask = (dow >= 5).astype(float)
demand = demand + 10.0 * weekend_mask

target = pd.Series(demand, index=dates, name='demand')
exog = pd.DataFrame({'is_weekend': weekend_mask}, index=dates)

X, y, feature_names = series_and_exogenous_to_supervised(target, exog, n_lags=7, horizon=1)

# Walk-forward evaluation
wf_results = walk_forward_evaluation(
    X,
    y,
    model_fn=rf_model_exog_fn,
    initial_train=200,
    step=30,
    horizon=30,
)

print(wf_results)

# Plot MAE and RMSE across walk-forward steps
plot_walk_forward_metrics(wf_results)

### 8. How to use Companion C in the time-series mini-project

For the time-series mini-project, you can use these templates in several ways:

- apply residual-based anomaly detection to a real or synthetic series;
- study concept drift by tracking performance over time on real or simulated data;
- explore multivariate dependence with cross-correlation and simple Granger-style tests;
- build ML models using lagged and exogenous/context features and examine feature importance.

Simulation-based investigations are explicitly acceptable. For
example:

- generate synthetic series with known drift or anomaly patterns (for example level shifts,
  variance changes, or regime-specific dynamics);
- apply the anomaly and drift templates to study how different methods respond;
- construct multivariate synthetic data with known cross-dependencies and test whether
  cross-correlation and Granger-style tools recover the intended structure.

A strong mini-project does not need complicated models. It needs:

- clear problem framing and data description (Companions A and B, Lectures TS1–TS3);
- honest comparison against simple baselines;
- explicit discussion of drift, anomalies, and multivariate structure where relevant;
- sceptical evaluation of model behaviour, including cases where baselines are hard to beat.

Companion C provides reusable code for anomaly, drift, multivariate
dependence, and ML-with-context workflows. You should adapt these
templates to your own data or simulations and document what works,
what does not, and why.